# PYTHON APPROFONDISSEMENT
### Exercices et Démos
###  

In [1]:
import psutil
print(psutil.virtual_memory())

svmem(total=3883982848, available=3174744064, percent=18.3, used=407642112, free=1109270528, active=1108324352, inactive=901656576, buffers=411213824, cached=1955856384, shared=6848512, slab=732487680)


In [17]:
Data_Path='/home/jupyter/EF-form-py-data/data/'

In [6]:
# Big vecteur numérique de 10 M 
import numpy as np
import pandas as pd
x=pd.DataFrame(np.random.rand(25000000))
y=pd.DataFrame(np.random.rand(25000000))

df=pd.concat([x,y],axis=1)
df.columns=["x","y"]


In [5]:
import psutil
print(psutil.virtual_memory())

svmem(total=3883982848, available=849125376, percent=78.1, used=2809995264, free=786489344, active=2883588096, inactive=145829888, buffers=55144448, cached=232353792, shared=6848512, slab=32677888)


In [7]:
print(df.dtypes)
df["z"]=df.x.apply(lambda x:1 if x>0.5 else 0)


In [8]:
%time df.describe()

CPU times: user 2.23 s, sys: 608 ms, total: 2.84 s
Wall time: 3.02 s


,x,y,z
count,2.500000e+07,2.500000e+07,2.500000e+07
mean,4.999931e-01,5.001417e-01,4.998739e-01
std,2.886515e-01,2.887348e-01,5.000000e-01
min,5.901001e-08,2.543606e-08,0.000000e+00
25%,2.501783e-01,2.501111e-01,0.000000e+00
50%,4.998745e-01,5.001604e-01,0.000000e+00
75%,7.499523e-01,7.502859e-01,1.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000000 entries, 0 to 24999999
Data columns (total 3 columns):
 #   Column  Dtype  
---  ------  -----  
 0   x       float64
 1   y       float64
 2   z       int64  
dtypes: float64(2), int64(1)
memory usage: 572.2 MB


In [26]:
import psutil
print(psutil.virtual_memory())

svmem(total=8415277056, available=5042847744, percent=40.1, used=3372429312, free=5042847744)


In [11]:
import os
os.getcwd()

'/home/jupyter/EF-form-py-data/notebooks'

In [10]:
# écrire sur disque le gros fichier : attention c 'est long avec 100M
df.to_csv('../data/bigfile25M.csv')
# attention 100M x 2 ca fait presque 10GO

FileNotFoundError: [Errno 2] No such file or directory: '/data/bigfile25M.csv'

In [29]:
%%timeit
df.describe()

1.21 s ± 19.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# Demo Lecture Partielle

In [12]:
import pandas as pd

#################################
#  Restriction lignes
#################################
#Data_Path = "C:\\Users\\dgr\\Documents\\data\\"

df = pd.read_csv(Data_Path+"bigfile25M.csv",nrows=100,sep=',')
df.shape
df.columns

Index(['Unnamed: 0', 'x', 'y', 'z'], dtype='object')

In [31]:

df = pd.read_csv(Data_Path+"bigfile25M.csv", skiprows=9000000, sep=',')
df.shape


(1000000, 4)

In [34]:
#################################
#  Restriction colonnes
#################################
use_cols =['x', 'y'] # Selection des colonnes
col_type={'x':float, 'y':float}  # typage des colonnes à lire
# Lecture

df=pd.read_csv(Data_Path+"bigfile25M.csv", nrows=1000000, usecols= use_cols, dtype= col_type, sep=',')
df.shape


(1000000, 2)

# Demo_Chunk

In [18]:
import pandas as pd

data_iterator = pd.read_csv(Data_Path+"bigfile25M.csv", chunksize=1000000)


In [14]:
# lecture du fichier par morceau et a chaque morceau on fait un filtre ce qui permet à la fin
# de disposer d'un fichier sans avoir tout chargé en mémoire d'un coup
chunk_list = []  

# Each chunk is in dataframe format
for data_chunk in data_iterator:
    print(data_chunk.shape)
    filtered_chunk = data_chunk[data_chunk.z==1]
    chunk_list.append(filtered_chunk)

filtered_data = pd.concat(chunk_list)
filtered_data.shape

(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)
(1000000, 4)


(12503300, 4)

# Demo Dask

In [19]:
#Data_Path = "C:\\Users\\dgr\\Documents\\data\\"
# dask divise aussi en bloc mais utilise tous les coeurs et donc processe plusieurs blocs
import dask.dataframe as dd
dask_df = dd.read_csv(Data_Path+"bigfile25M.csv") 

# voir les meta données
dask_df 


,Unnamed: 0,x,y,z
npartitions=20,,,,
,int64,float64,float64,int64
,...,...,...,...
...,...,...,...,...
,...,...,...,...
,...,...,...,...


In [4]:
dask_df.columns
dask_df.head() 

,Unnamed: 0,x,y,z
0,0,0.882637,0.517809,1
1,1,0.578324,0.111232,1
2,2,0.851462,0.607887,1
3,3,0.581744,0.995462,1
4,4,0.122080,0.659044,0


In [18]:
# Les opérations sont réalisées par bloc ce qui permet d'éviter de remplir la mémoire
# Efficace pour la mémoire mais ralentit l'exécution par rapport à pandas
%time xmoy = dask_df.groupby('z').x.mean()


Wall time: 14.6 ms


Dask Series Structure:
npartitions=1
    float64
        ...
Name: x, dtype: float64
Dask Name: truediv, 69 tasks

In [17]:
# déclencher le calcul
%time xm = xmoy.compute()
xm

Wall time: 10.1 s


z
0    0.250079
1    0.750040
Name: x, dtype: float64

In [19]:
# avec pandas :plus rapide car charge en memoire
%time df.groupby('z').x.mean()

Wall time: 2.56 ms


z
0    0.234545
1    0.762215
Name: x, dtype: float64

In [8]:
%time NewDF = dask_df[dask_df.z == 0].y + 100
%time NewDF = dask_df.compute()

Wall time: 0 ns
Wall time: 2.38 s


In [20]:
# Enr le fichier en parquet  format de fichier orienté colonne plus rapide à lire et à écrire
dask_df.to_parquet(Data_Path+'bigf25M.parquet', engine='pyarrow')

In [21]:
parquet_df = dd.read_parquet(Data_Path+'bigf25M.parquet', engine='pyarrow')
parquet_df

,Unnamed: 0,x,y,z
npartitions=20,,,,
,int64,float64,float64,int64
,...,...,...,...
...,...,...,...,...
,...,...,...,...
,...,...,...,...


In [22]:
%time parquet_df.groupby('z').x.mean().compute()
# deux fois plus rapide

Wall time: 5.21 s


z
0    0.250079
1    0.750040
Name: x, dtype: float64